In [3]:
import asyncio
import google.generativeai as genai
from concurrent.futures import ThreadPoolExecutor
import nest_asyncio
nest_asyncio.apply()  # For Jupyter/Colab
import numpy as np
from dotenv import load_dotenv
import os
import pandas as pd
from tqdm.asyncio import tqdm
import time
import numpy as np
import json


import asyncio
import json
import time
import os
from concurrent.futures import ThreadPoolExecutor
from typing import Dict, List, Optional
import google.generativeai as genai
load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
genai.configure(api_key=GEMINI_API_KEY)


In [16]:
# Create multiple model instances for parallel calls
def create_gemini_model():
    """Create Gemini model with JSON response schema"""
    import google.generativeai as genai
    
    generation_config = {
        "temperature": 0.1,  # Very low for lite model consistency
        "response_mime_type": "application/json",
 
    }
    
    model = genai.GenerativeModel(
        model_name='gemini-2.0-flash',
        generation_config=generation_config
    )
    
    return model

In [35]:
class NoseCheck:
    """
    LLM-as-Judge reward function for GRPO training.
    Returns multiple granular scores per completion for flexible reward shaping.
    
    Usage in GRPO:
        judge = NoseCheck(judge_prompts, batch_eval=True, responses_per_prompt=4)
        
        def combined_reward(prompts, completions, domain, **kwargs):
            # Get granular scores from judge
            scores_dict = judge(prompts, completions, domain, **kwargs)
            
            # Your custom reward calculation
            rewards = []
            for i in range(len(completions)):
                reward = (
                    scores_dict['reasoning'][i] * 0.3 +
                    scores_dict['correctness'][i] * 0.5 +
                    scores_dict['format'][i] * 0.2
                )
                rewards.append(reward)
            return rewards
        
        grpo_trainer = GRPOLearner(
            rl_cluster=rl_cluster,
            reward_fns=[combined_reward],
            grpo_config=grpo_config,
        )
    """
    
    def __init__(self, judge_prompts: Dict[str, str], max_concurrent: int = 30, 
                 batch_eval: bool = False, responses_per_prompt: int = 4,
                 score_keys: List[str] = None):
        """
        Args:
            judge_prompts: Dict mapping domain -> prompt template string
            max_concurrent: Max parallel API calls (Gemini 2.0 Flash: 2K RPM)
            batch_eval: If True, judge all responses per prompt in one call
            responses_per_prompt: Number of responses per prompt in GRPO
            score_keys: List of score keys to expect from LLM (default: auto-detect from first response)
        """
        self.domain_prompts = judge_prompts
        self.max_concurrent = max_concurrent
        self.batch_eval = batch_eval
        self.responses_per_prompt = responses_per_prompt
        self.score_keys = score_keys  # e.g., ['reasoning', 'correctness', 'format', 'hallucination']
        self.semaphore = asyncio.Semaphore(max_concurrent)
        self.executor = ThreadPoolExecutor(max_workers=max_concurrent)
        
        # Metrics
        self.call_count = 0
        self.total_time = 0
    
    def __call__(self, prompts: List[str], completions: List[str], 
                 domain: List[str], ground_truth: Optional[List[str]] = None, 
                 **kwargs) -> Dict[str, List[float]]:
        """
        Main reward function called by GRPO trainer.
        
        Args:
            prompts: List of N prompts
            completions: List of N*responses_per_prompt completions (flattened)
            domain: List of N domains (one per prompt)
            ground_truth: Optional list of N ground truth answers
            **kwargs: Additional fields from dataset
        
        Returns:
            Dict[str, List[float]]: Dictionary of score lists
            {
                'reasoning': [s1, s2, ..., sN*responses_per_prompt],
                'correctness': [s1, s2, ..., sN*responses_per_prompt],
                'format': [s1, s2, ..., sN*responses_per_prompt],
                'hallucination': [s1, s2, ..., sN*responses_per_prompt],
            }
            Each score normalized to 0-1 range
        """
        self.call_count = 0
        start_time = time.time()
        
        # Validate input lengths
        expected_completions = len(prompts) * self.responses_per_prompt
        if len(completions) != expected_completions:
            print(f"WARNING: Expected {expected_completions} completions, got {len(completions)}")
            print(f"  Prompts: {len(prompts)}, Responses/prompt: {self.responses_per_prompt}")
        
        scores_dict = asyncio.run(self._evaluate_batch(
            prompts, completions, domain, ground_truth, **kwargs
        ))
        
        elapsed = time.time() - start_time
        self.total_time += elapsed
        
        # Log metrics
        self._log_metrics(len(prompts), len(completions), elapsed, scores_dict)
        
        return scores_dict
    
    def _log_metrics(self, num_prompts: int, num_completions: int, 
                     elapsed: float, scores_dict: Dict[str, List[float]]):
        """Log evaluation metrics"""
        print(f"\n{'='*70}")
        print(f"🤥 NoseCheck Complete")
        print(f"{'='*70}")
        print(f"Mode: {'BATCH' if self.batch_eval else 'SINGLE'} "
              f"({self.responses_per_prompt} responses/prompt)")
        print(f"Prompts: {num_prompts} | Completions: {num_completions}")
        print(f"API Calls: {self.call_count}")
        print(f"Time: {elapsed:.2f}s | Rate: {num_completions/elapsed:.1f} evals/sec")
        
        # Show stats for each score type
        print(f"\nScore Statistics:")
        for key, scores in scores_dict.items():
            avg = sum(scores) / len(scores)
            print(f"  {key:15s}: avg={avg:.3f}, min={min(scores):.3f}, max={max(scores):.3f}")
        print(f"{'='*70}\n")
    
    async def _evaluate_batch(self, prompts, completions, domains, ground_truths, **kwargs):
        """Route to appropriate evaluation mode"""
        if self.batch_eval:
            return await self._batch_eval_mode(prompts, completions, domains, ground_truths)
        else:
            return await self._single_eval_mode(prompts, completions, domains, ground_truths)
    
    async def _single_eval_mode(self, prompts, completions, domains, ground_truths):
        """One LLM call per completion - more flexible, easier debugging"""
        # Expand prompt-level data to match completions
        expanded_prompts = [p for p in prompts for _ in range(self.responses_per_prompt)]
        expanded_domains = [d for d in domains for _ in range(self.responses_per_prompt)]
        expanded_gt = [gt for gt in (ground_truths or [None]*len(prompts)) 
                       for _ in range(self.responses_per_prompt)]
        
        tasks = [
            self._judge_single(p, c, d, gt)
            for p, c, d, gt in zip(expanded_prompts, completions, 
                                   expanded_domains, expanded_gt)
        ]
        
        results = await asyncio.gather(*tasks)
        
        # Convert list of dicts to dict of lists
        # results = [{'reasoning': 0.8, 'correctness': 0.9}, ...]
        # -> {'reasoning': [0.8, ...], 'correctness': [0.9, ...]}
        if not results:
            return {}
        
        keys = results[0].keys()
        return {key: [r[key] for r in results] for key in keys}
    
    async def _batch_eval_mode(self, prompts, completions, domains, ground_truths):
        """One LLM call per N responses - faster, fairer, cheaper"""
        # Group completions by prompt
        grouped_completions = [
            completions[i*self.responses_per_prompt:(i+1)*self.responses_per_prompt]
            for i in range(len(prompts))
        ]
        
        tasks = [
            self._judge_group(p, group, d, gt)
            for p, group, d, gt in zip(
                prompts, grouped_completions, domains,
                ground_truths or [None]*len(prompts)
            )
        ]
        
        results = await asyncio.gather(*tasks)
        
        # results = [
        #   [{'reasoning': 0.8, 'correctness': 0.9}, ...],  # 4 dicts for prompt 1
        #   [{'reasoning': 0.7, 'correctness': 0.8}, ...],  # 4 dicts for prompt 2
        # ]
        # Flatten and convert to dict of lists
        flattened = [score_dict for group in results for score_dict in group]
        
        if not flattened:
            return {}
        
        keys = flattened[0].keys()
        return {key: [r[key] for r in flattened] for key in keys}
    
    async def _judge_single(self, prompt: str, completion: str, 
                           domain: str, ground_truth: Optional[str]) -> Dict[str, float]:
        """Judge one completion, returns dict of scores"""
        async with self.semaphore:
            self.call_count += 1
            
            judge_prompt = self.domain_prompts[domain].format(
                prompt=prompt,
                completion=completion,
                ground_truth=ground_truth or "N/A"
            )
            
            # Debug first call
            if self.call_count == 1:
                print(f"\n[DEBUG] First Judge Prompt Sent:")
                print(f"{judge_prompt[:500]}...")
                print(f"[DEBUG] End of prompt\n")
            
            try:
                result = await self._call_gemini(judge_prompt)
                
                # Debug first result
                if self.call_count == 1:
                    print(f"\n[DEBUG] Parsed result: {result}")
                    print(f"[DEBUG] Score keys: {list(result.keys())}\n")
                
                # Normalize all scores to 0-1 range
                normalized = {key: val / 10.0 for key, val in result.items() 
                             if isinstance(val, (int, float))}
                
                return normalized
            except Exception as e:
                print(f"❌ Error judging single completion: {e}")
                import traceback
                traceback.print_exc()
                # Return zeros for all expected keys
                if self.score_keys:
                    return {key: 0.0 for key in self.score_keys}
                return {'score': 0.0}
    
    async def _judge_group(self, prompt: str, completions_group: List[str],
                          domain: str, ground_truth: Optional[str]) -> List[Dict[str, float]]:
        """Judge N responses in one call, returns list of score dicts"""
        async with self.semaphore:
            self.call_count += 1
            
            # Format all responses
            responses_text = "\n\n".join([
                f"Response {i+1}:\n{comp}"
                for i, comp in enumerate(completions_group)
            ])
            
            judge_prompt = self.domain_prompts[domain].format(
                prompt=prompt,
                completions=responses_text,
                ground_truth=ground_truth or "N/A",
                num_responses=len(completions_group)
            )
            
            try:
                result = await self._call_gemini(judge_prompt)
                
                # Expected format: {"scores": [{"reasoning": 8, "correctness": 9}, ...]}
                scores_list = result.get("scores", [])
                
                if not scores_list:
                    # Fallback: return zeros
                    if self.score_keys:
                        return [{key: 0.0 for key in self.score_keys} 
                                for _ in completions_group]
                    return [{'score': 0.0} for _ in completions_group]
                
                # Normalize to 0-1 range
                normalized_list = []
                for score_dict in scores_list:
                    normalized = {key: val / 10.0 for key, val in score_dict.items()
                                 if isinstance(val, (int, float))}
                    normalized_list.append(normalized)
                
                return normalized_list
                
            except Exception as e:
                print(f"❌ Error judging group: {e}")
                import traceback
                traceback.print_exc()
                # Return zeros
                if self.score_keys:
                    return [{key: 0.0 for key in self.score_keys} 
                            for _ in completions_group]
                return [{'score': 0.0} for _ in completions_group]
    
    async def _call_gemini(self, prompt: str) -> dict:
        """Async wrapper for Gemini API call"""
        loop = asyncio.get_event_loop()
        result = await loop.run_in_executor(
            self.executor,
            self._blocking_gemini_call,
            prompt
        )
        return result
    
    def _blocking_gemini_call(self, prompt: str) -> dict:
        """Blocking Gemini API call"""
        try:
            model = create_gemini_model()
            response = model.generate_content(prompt)
            
            # Debug: print first response to see what we're getting
            if self.call_count == 1:
                print(f"\n[DEBUG] First Gemini Response:")
                print(f"Response text: {response.text}")
                print(f"[DEBUG] End of response\n")
            
            return json.loads(response.text)
        except json.JSONDecodeError as e:
            print(f"\n❌ JSON decode error: {e}")
            print(f"Response text: {response.text[:500]}")
            return {"score": 0}
        except Exception as e:
            print(f"\n❌ Gemini API error: {e}")
            print(f"Error type: {type(e).__name__}")
            import traceback
            traceback.print_exc()
            return {"score": 0}

In [ ]:

# SINGLE_MODE_PROMPTS = {
#     "math": """You are evaluating a mathematical reasoning response.

# Question: {prompt}

# Student's Solution:
# {completion}

# Correct Answer: {ground_truth}

# Evaluate this solution and return a JSON score from 0-10 based on:
# 1. Correctness of final answer (60% weight)
# 2. Quality and clarity of reasoning steps (30% weight)
# 3. Proper mathematical notation (10% weight)

# Return JSON format:
# {{"score": X}}

# Where X is 0-10 (decimals allowed, e.g., 7.5)""",
    
#     "coding": """You are evaluating a coding solution.

# Problem: {prompt}

# Student's Code:
# {completion}

# Expected Behavior: {ground_truth}

# Evaluate this code and return a JSON score from 0-10 based on:
# 1. Correctness - does it solve the problem? (50% weight)
# 2. Code quality - readability, efficiency (30% weight)
# 3. Edge case handling (20% weight)

# Return JSON format:
# {{"score": X}}

# Where X is 0-10 (decimals allowed)""",
    
#     "creative_writing": """You are evaluating creative writing.

# Prompt: {prompt}

# Response:
# {completion}

# Evaluate this creative writing and return a JSON score from 0-10 based on:
# 1. Creativity and originality (40% weight)
# 2. Coherence and structure (30% weight)
# 3. Engagement and style (30% weight)

# Return JSON format:
# {{"score": X}}

# Where X is 0-10 (decimals allowed)"""
# }

# BATCH_MODE_PROMPTS = {
#     "math": """You are evaluating {num_responses} mathematical reasoning responses to the same question.

# Question: {prompt}

# Correct Answer: {ground_truth}

# {completions}

# Evaluate each solution fairly and consistently. Score each from 0-10 based on:
# 1. Correctness of final answer (60% weight)
# 2. Quality and clarity of reasoning (30% weight)
# 3. Mathematical notation (10% weight)

# Return JSON with array of {num_responses} scores:
# {{"scores": [s1, s2, ..., s{num_responses}]}}

# Where each score is 0-10 (decimals allowed)""",
    
#     "coding": """You are evaluating {num_responses} coding solutions to the same problem.

# Problem: {prompt}

# Expected Behavior: {ground_truth}

# {completions}

# Evaluate each solution fairly. Score each from 0-10 based on:
# 1. Correctness (50% weight)
# 2. Code quality (30% weight)
# 3. Edge cases (20% weight)

# Return JSON with array of {num_responses} scores:
# {{"scores": [s1, s2, ..., s{num_responses}]}}

# Where each score is 0-10 (decimals allowed)""",
    
#     "creative_writing": """You are evaluating {num_responses} creative writing responses to the same prompt.

# Prompt: {prompt}

# {completions}

# Evaluate each response fairly. Score each from 0-10 based on:
# 1. Creativity and originality (40% weight)
# 2. Coherence and structure (30% weight)
# 3. Engagement and style (30% weight)

# Return JSON with array of {num_responses} scores:
# {{"scores": [s1, s2, ..., s{num_responses}]}}

# Where each score is 0-10 (decimals allowed)"""
# }

In [40]:
SINGLE_MODE_PROMPTS = {
    "math": """You are evaluating a mathematical reasoning response.

Question: {prompt}

Student's Solution:
{completion}

Correct Answer: {ground_truth}

Evaluate this solution and return JSON with multiple scores (each 0-10):

1. **reasoning** (0-10): Quality and clarity of reasoning steps
   - Clear logical flow
   - Proper mathematical reasoning
   - Step-by-step breakdown

2. **correctness** (0-10): Accuracy of the final answer
   - Matches ground truth
   - Mathematically correct
   - Appropriate precision

3. **format** (0-10): Presentation quality
   - Proper notation
   - Clear structure
   - Well-formatted

4. **hallucination** (0-10): Factual accuracy (10=no hallucinations)
   - No made-up facts
   - No incorrect formulas
   - Stays grounded to the problem

Return JSON format:
{{
  "reasoning": X,
  "correctness": Y,
  "format": Z,
  "hallucination": W
}}

All values 0-10 (decimals allowed)""",
    
    "coding": """You are evaluating a coding solution.

Problem: {prompt}

Student's Code:
{completion}

Expected Behavior: {ground_truth}

Evaluate this code and return JSON with multiple scores (each 0-10):

1. **reasoning** (0-10): Problem-solving approach
   - Logical algorithm choice
   - Clear thought process
   - Good strategy

2. **correctness** (0-10): Code functionality
   - Solves the problem
   - Handles edge cases
   - No bugs

3. **format** (0-10): Code quality
   - Readability
   - Clean structure
   - Good practices

4. **hallucination** (0-10): Technical accuracy (10=no hallucinations)
   - No non-existent functions
   - Valid syntax
   - Correct API usage

Return JSON format:
{{
  "reasoning": X,
  "correctness": Y,
  "format": Z,
  "hallucination": W
}}

All values 0-10 (decimals allowed)""",
}

BATCH_MODE_PROMPTS = {
    "math": """You are evaluating {num_responses} mathematical reasoning responses to the same question.

Question: {prompt}

Correct Answer: {ground_truth}

{completions}

Evaluate each solution fairly and return JSON with array of score objects.

Each response gets 4 scores (0-10):
1. **reasoning**: Quality of reasoning steps
2. **correctness**: Accuracy of final answer  
3. **format**: Presentation quality
4. **hallucination**: Factual accuracy (10=no hallucinations)

Return JSON format:
{{
  "scores": [
    {{"reasoning": X1, "correctness": Y1, "format": Z1, "hallucination": W1}},
    {{"reasoning": X2, "correctness": Y2, "format": Z2, "hallucination": W2}},
    ...
  ]
}}

All values 0-10 (decimals allowed)""",
    
    "coding": """You are evaluating {num_responses} coding solutions to the same problem.

Problem: {prompt}

Expected Behavior: {ground_truth}

{completions}

Evaluate each solution fairly and return JSON with array of score objects.

Each response gets 4 scores (0-10):
1. **reasoning**: Problem-solving approach
2. **correctness**: Code functionality
3. **format**: Code quality and readability
4. **hallucination**: Technical accuracy (10=no hallucinations)

Return JSON format:
{{
  "scores": [
    {{"reasoning": X1, "correctness": Y1, "format": Z1, "hallucination": W1}},
    {{"reasoning": X2, "correctness": Y2, "format": Z2, "hallucination": W2}},
    ...
  ]
}}

All values 0-10 (decimals allowed)""",
}

In [ ]:
def create_sample_data(batch_size=4, responses_per_prompt=4):
    """
    Create sample data as GRPO would pass it
    
    Returns:
        prompts: [p1, p2, ..., pN]
        completions: [c1_1, c1_2, c1_3, c1_4, c2_1, ...] (flattened)
        domains: [d1, d2, ..., dN]
        ground_truths: [gt1, gt2, ..., gN]
    """
    
    prompts = [
        "Solve: If a train travels 120 miles in 2 hours, what is its average speed?",
        "Write a Python function to reverse a string without using built-in reverse.",
        "Calculate: What is 15% of 240?",
        "Write a function that checks if a number is prime."
    ]
    
    # 4 responses per prompt
    completions = [
        # Prompt 1 responses (math)
        "<reasoning>Speed = Distance / Time = 120 / 2 = 60</reasoning><answer>60 mph</answer>",
        "<reasoning>Distance is 120, time is 2. Speed = 120/2 = 60 mph</reasoning><answer>60 miles per hour</answer>",
        "<reasoning>Average speed means total distance over total time. 120 miles / 2 hours = 60</reasoning><answer>60</answer>",
        "<reasoning>Speed formula: s = d/t. So 120/2 = 60 mph</reasoning><answer>The speed is 60 mph</answer>",
        
        # Prompt 2 responses (coding)
        "<reasoning>Use slicing with step -1</reasoning><answer>def reverse_string(s):\n    return s[::-1]</answer>",
        "<reasoning>Build new string by iterating backwards</reasoning><answer>def reverse_string(s):\n    result = ''\n    for i in range(len(s)-1, -1, -1):\n        result += s[i]\n    return result</answer>",
        "<reasoning>Use recursion</reasoning><answer>def reverse_string(s):\n    if len(s) <= 1:\n        return s\n    return reverse_string(s[1:]) + s[0]</answer>",
        "<reasoning>Two pointer swap</reasoning><answer>def reverse_string(s):\n    chars = list(s)\n    left, right = 0, len(chars)-1\n    while left < right:\n        chars[left], chars[right] = chars[right], chars[left]\n        left += 1\n        right -= 1\n    return ''.join(chars)</answer>",
        
        # Prompt 3 responses (math)
        "<reasoning>15% means 15/100. So (15/100) * 240 = 36</reasoning><answer>36</answer>",
        "<reasoning>0.15 * 240 = 36</reasoning><answer>36</answer>",
        "<reasoning>240 * 15% = 240 * 0.15 = 35</reasoning><answer>35</answer>",  # Wrong!
        "<reasoning>15 percent of 240: (15/100)*240 = 3600/100 = 36</reasoning><answer>36</answer>",
        
        # Prompt 4 responses (coding)
        "<reasoning>Check divisibility from 2 to sqrt(n)</reasoning><answer>def is_prime(n):\n    if n < 2:\n        return False\n    for i in range(2, int(n**0.5)+1):\n        if n % i == 0:\n            return False\n    return True</answer>",
        "<reasoning>Check all numbers up to n</reasoning><answer>def is_prime(n):\n    if n < 2:\n        return False\n    for i in range(2, n):\n        if n % i == 0:\n            return False\n    return True</answer>",
        "<reasoning>Check divisibility</reasoning><answer>def is_prime(n):\n    return n > 1 and all(n % i != 0 for i in range(2, n))</answer>",
        "<reasoning>Optimized prime check</reasoning><answer>def is_prime(n):\n    if n <= 1:\n        return False\n    if n == 2:\n        return True\n    if n % 2 == 0:\n        return False\n    for i in range(3, int(n**0.5)+1, 2):\n        if n % i == 0:\n            return False\n    return True</answer>",
    ]
    
    domains = ["math", "coding", "math", "coding"]
    
    ground_truths = [
        "60 mph",
        "Function should reverse string without using built-in reverse",
        "36",
        "Function should return True for prime numbers, False otherwise"
    ]
    
    return prompts, completions, domains, ground_truths


def test_single_mode():
    """Test single evaluation mode (1 call per completion)"""
    print("\n" + "="*70)
    print("TEST 1: SINGLE MODE")
    print("="*70)
    
    judge = NoseCheck(
        judge_prompts=SINGLE_MODE_PROMPTS,  # Changed from domain_prompts
        max_concurrent=30,
        batch_eval=False,
        responses_per_prompt=4
    )
    
    prompts, completions, domains, ground_truths = create_sample_data()
    
    print(f"\nInput:")
    print(f"  {len(prompts)} prompts × {judge.responses_per_prompt} responses = {len(completions)} completions")
    print(f"  Domains: {domains}")
    
    # Call exactly as GRPO would
    scores = judge(
        prompts=prompts,
        completions=completions,
        domain=domains,
        ground_truth=ground_truths
    )
    
    # Display results grouped by prompt
    print("\nResults by Prompt:")
    print("-" * 70)
    for i, (prompt, domain) in enumerate(zip(prompts, domains)):
        print(f"\nPrompt {i+1} [{domain}]: {prompt[:50]}...")
        for j in range(4):
            idx = i*4 + j
            print(f"  Response {j+1}:")
            for key in scores.keys():
                print(f"    {key:15s}: {scores[key][idx]:.3f}")
    
    return scores


def test_batch_mode():
    """Test batch evaluation mode (1 call per 4 completions)"""
    print("\n\n" + "="*70)
    print("TEST 2: BATCH MODE (Fairer + Faster)")
    print("="*70)
    
    judge = NoseCheck(
        judge_prompts=BATCH_MODE_PROMPTS,  # Changed from domain_prompts
        max_concurrent=32,
        batch_eval=True,
        responses_per_prompt=4
    )
    
    prompts, completions, domains, ground_truths = create_sample_data()
    
    print(f"\nInput:")
    print(f"  {len(prompts)} prompts × {judge.responses_per_prompt} responses = {len(completions)} completions")
    print(f"  Domains: {domains}")
    
    scores = judge(
        prompts=prompts,
        completions=completions,
        domain=domains,
        ground_truth=ground_truths
    )
    
    # Display results
    print("\nResults by Prompt:")
    print("-" * 70)
    for i, (prompt, domain) in enumerate(zip(prompts, domains)):
        print(f"\nPrompt {i+1} [{domain}]: {prompt[:50]}...")
        for j in range(4):
            idx = i*4 + j
            print(f"  Response {j+1}:")
            for key in scores.keys():
                print(f"    {key:15s}: {scores[key][idx]:.3f}")
    
    return scores


def custom_reward(scores):
    
    rewards = []
    domain='other'
    for i in range(len(scores['reasoning'])):
        # Custom weighting per domain
        if domain[i//4] == "math":
            reward = (
                scores['reasoning'][i] * 0.2 +
                scores['correctness'][i] * 0.6 +
                scores['format'][i] * 0.1 +
                scores['hallucination'][i] * 0.1
            )
        else:  # coding
            reward = (
                scores['reasoning'][i] * 0.3 +
                scores['correctness'][i] * 0.4 +
                scores['format'][i] * 0.2 +
                scores['hallucination'][i] * 0.1
            )
        rewards.append(reward)
    return rewards

In [38]:
# single_scores = test_single_mode()

In [42]:
batch_scores = test_batch_mode()
print(f"Batch Mode:  {len(batch_scores['reasoning'])} completions evaluated")



TEST 2: BATCH MODE (Fairer + Faster)

Input:
  4 prompts × 4 responses = 16 completions
  Domains: ['math', 'coding', 'math', 'coding']

🤥 NoseCheck Complete
Mode: BATCH (4 responses/prompt)
Prompts: 4 | Completions: 16
API Calls: 4
Time: 1.47s | Rate: 10.8 evals/sec

Score Statistics:
  reasoning      : avg=0.931, min=0.700, max=1.000
  correctness    : avg=0.938, min=0.000, max=1.000
  format         : avg=0.969, min=0.700, max=1.000
  hallucination  : avg=0.750, min=0.000, max=1.000


Results by Prompt:
----------------------------------------------------------------------

Prompt 1 [math]: Solve: If a train travels 120 miles in 2 hours, wh...
  Response 1:
    reasoning      : 1.000
    correctness    : 1.000
    format         : 1.000
    hallucination  : 1.000
  Response 2:
    reasoning      : 1.000
    correctness    : 1.000
    format         : 1.000
    hallucination  : 1.000
  Response 3:
    reasoning      : 0.800
    correctness    : 1.000
    format         : 0.700
    

In [45]:
custom_reward(batch_scores)

[0.9999999999999999,
 0.9999999999999999,
 0.88,
 0.9999999999999999,
 0.8999999999999999,
 0.8999999999999999,
 0.8999999999999999,
 0.8999999999999999,
 0.9999999999999999,
 0.9999999999999999,
 0.54,
 0.9999999999999999,
 0.9700000000000001,
 0.91,
 0.87,
 0.9999999999999999]

In [ ]:
from tqdm import tqdm
import numpy as np
from collections import defaultdict


def evaluate_with_nosecheck(
    dataset,
    sampler,
    judge,
    temperature=0.7,
    top_k=50,
    top_p=0.95,
    num_passes=1,
    save_responses=False,
    save_failures_only=False,
):
    """
    Evaluate model using NoseCheck judge.
    
    Args:
        dataset: Batches with 'input', 'answer', 'domain' fields
        sampler: Your model sampler
        judge: NoseCheck instance
        num_passes: Generate N responses per question, keep best
        save_responses: Save responses for analysis
        save_failures_only: Only save low-scoring responses
        
    Returns:
        (overall_stats, domain_stats, response_list)
        
    Usage:
        judge = NoseCheck(judge_prompts=BATCH_MODE_PROMPTS, batch_eval=True, 
                         responses_per_prompt=num_passes)
        stats, domain_stats, responses = evaluate_with_nosecheck(
            test_dataset, sampler, judge, num_passes=3
        )
    """
    
    # Accumulators
    overall_scores = defaultdict(list)
    domain_scores = defaultdict(lambda: defaultdict(list))
    response_list = []
    total = 0

    for batch in tqdm(dataset):
        questions = batch["input"]
        answers = batch.get("answer", [None] * len(questions))
        domains = batch["domain"]

        # Generate num_passes responses per question
        multiple_responses = [[] for _ in range(len(questions))]
        for p in range(num_passes):
            responses = generate(
                questions, sampler, temperature, top_k, top_p, seed=p
            )
            for idx, response in enumerate(responses):
                multiple_responses[idx].append(response)

        # Prepare for batch judging: flatten to [q1,q1,q1,q2,q2,q2,...]
        prompts_flat = []
        completions_flat = []
        domains_flat = []
        answers_flat = []
        
        for question, responses, domain, answer in zip(
            questions, multiple_responses, domains, answers
        ):
            for response in responses:
                prompts_flat.append(question)
                completions_flat.append(response)
                domains_flat.append(domain)
                answers_flat.append(answer)

        # Get all scores from judge in one batch call
        judge_scores = judge(
            prompts=prompts_flat,
            completions=completions_flat,
            domain=domains_flat,
            ground_truth=answers_flat
        )
        
        # Process each question
        for q_idx, (question, responses, domain, answer) in enumerate(
            zip(questions, multiple_responses, domains, answers)
        ):
            # Extract scores for this question's responses
            start_idx = q_idx * num_passes
            end_idx = (q_idx + 1) * num_passes
            
            question_judge_scores = {
                key: scores[start_idx:end_idx]
                for key, scores in judge_scores.items()
            }
            
            # Find best response (highest combined score)
            combined = [
                sum(question_judge_scores[key][i] for key in question_judge_scores)
                for i in range(num_passes)
            ]
            best_idx = np.argmax(combined)
            best_response = responses[best_idx]
            best_scores = {
                key: question_judge_scores[key][best_idx]
                for key in question_judge_scores
            }
            
            # Record scores
            for key, score in best_scores.items():
                overall_scores[key].append(score)
                domain_scores[domain][key].append(score)
            
            # Save if requested
            if save_responses:
                if save_failures_only:
                    # Only save if correctness is low
                    if best_scores.get('correctness', 1.0) < 0.5:
                        response_list.append((
                            question, answer, responses, best_scores, domain
                        ))
                else:
                    response_list.append((
                        question, answer, responses, best_scores, domain
                    ))
            
            total += 1
            
            # Progress update
            if total % 10 == 0:
                print(f"\n===> {total=}")
                for key in overall_scores:
                    avg = np.mean(overall_scores[key])
                    print(f"  {key}: {avg:.3f}")

    # Compile final stats
    print("\n" + "="*70)
    print("OVERALL RESULTS")
    print("="*70)
    print(f"Total: {total}")
    
    overall_stats = {}
    for key, scores in overall_scores.items():
        overall_stats[key] = {
            'mean': np.mean(scores),
            'std': np.std(scores),
            'min': np.min(scores),
            'max': np.max(scores),
        }
        print(f"{key:15s}: {overall_stats[key]['mean']:.3f} ± {overall_stats[key]['std']:.3f}")
    
    print("\n" + "="*70)
    print("BY DOMAIN")
    print("="*70)
    
    domain_stats = {}
    for domain, scores_dict in domain_scores.items():
        n = len(scores_dict[list(scores_dict.keys())[0]])
        print(f"\n{domain} (n={n}):")
        domain_stats[domain] = {}
        for key, scores in scores_dict.items():
            domain_stats[domain][key] = {
                'mean': np.mean(scores),
                'std': np.std(scores),
            }
            print(f"  {key:15s}: {domain_stats[domain][key]['mean']:.3f}")

    return overall_stats, domain_stats, response_list


def generate(questions, sampler, temperature, top_k, top_p, seed=0):
    """
    Generate responses - replace with your actual sampler call.
    
    Example for Tunix:
        return sampler.generate(
            prompts=questions,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            max_decode_steps=TOTAL_GENERATION_STEPS,
            seed=seed
        )
    """
    raise NotImplementedError("Implement with your sampler")


# Quick evaluation wrapper
def quick_eval(dataset, sampler, judge):
    """Quick evaluation with defaults"""
    return evaluate_with_nosecheck(
        dataset=dataset,
        sampler=sampler,
        judge=judge,
        num_passes=1,
        save_responses=False
    )



# def batch_iterator(data, batch_size=32):
#     """Convert list/dataframe to batch iterator"""
#     for i in range(0, len(data), batch_size):
#         batch_data = data[i:i+batch_size]
        
#         # Create batch dict
#         batch = {
#             "input": [item["input"] for item in batch_data],
#             "answer": [item["answer"] for item in batch_data],
#             "domain": [item["domain"] for item in batch_data],
#         }
#         yield batch

# # Usage
# my_data = [
#     {"input": "What is 2+2?", "answer": "4", "domain": "math"},
#     {"input": "Write hello world", "answer": "print('hello')", "domain": "coding"},
#     # ... more samples
# ]

# dataset = batch_iterator(my_data, batch_size=32)

# # Now you can iterate
# for batch in dataset:
#     questions = batch["input"]  # List of up to 32 questions

In [ ]:
judge = NoseCheck(
    judge_prompts=BATCH_MODE_PROMPTS,
    batch_eval=True,
    responses_per_prompt=3,  # Must match num_passes
    max_concurrent=30
)

# Evaluate
overall, by_domain, failures = evaluate_with_nosecheck(
    dataset=test_dataset,
    sampler=your_sampler,
    judge=judge,
    num_passes=3,
    save_responses=True,
    save_failures_only=True
)

# Analyze
print(f"Reasoning: {overall['reasoning']['mean']:.3f}")
print(f"Correctness: {overall['correctness']['mean']:.3f}")

for domain, stats in by_domain.items():
    print(f"\n{domain}:")
    print(f"  Correctness: {stats['correctness']['mean']:.3f}")

# Check failures
for question, answer, responses, scores, domain in failures:
    print(f"\nFailed [{domain}]: {question}")
    print(f"Expected: {answer}")
    print(f"Scores: {scores}")